# Import and Load data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
df = pd.read_csv('hotel_bookings.csv')

In [2]:
df.shape      # how many rows and columns

(119390, 32)

In [3]:
df.head() # first 5 rows

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [4]:
df.info() #column types

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [5]:
df.isnull().sum()  # check for missing values

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

# Clean the data

In [6]:
# Filling missing values 

df['children'] = df['children'].fillna(0).infer_objects(copy=False) 

In [7]:
df['country']= df['country'].fillna('Unknown')

In [8]:
# For company and agent - blank means "Direct/Individual booking"

df['is_company_booking'] = df['company'].notnull().astype(int)  # 1 = company, 0 = individual
df['is_agent_booking'] = df['agent'].notnull().astype(int)      # 1 = via agent, 0 = direct

# Drop the original company column as it's not useful
df.drop(columns=['company'], inplace=True)

print("Company bookings:", df['is_company_booking'].sum())
print("Agent bookings:", df['is_agent_booking'].sum())
print("Direct bookings:", (df['is_agent_booking'] == 0).sum())

Company bookings: 6797
Agent bookings: 103050
Direct bookings: 16340


In [9]:
# Remove rows where people made a booking with 0 guests (data error)
df = df[df['adults'] + df['children'] + df['babies'] > 0]

In [10]:
# Create a total revenue column (average daily rate x total nights stayed)
df['total_nights']= df['stays_in_weekend_nights']+ df['stays_in_week_nights']
df['total_revenue']= df['total_nights'] * df['adr']  

In [11]:
# Convert arrival date into a proper date column

df['arrival_date'] = pd.to_datetime(df['arrival_date_year'].astype(str) + '-' +df['arrival_date_month'].astype(str)+ '-'+ df['arrival_date_day_of_month'].astype(str))

print("Clean data shape:", df.shape)

Clean data shape: (119210, 36)


In [12]:
df['arrival_date']

0        2015-07-01
1        2015-07-01
2        2015-07-01
3        2015-07-01
4        2015-07-01
            ...    
119385   2017-08-30
119386   2017-08-31
119387   2017-08-31
119388   2017-08-31
119389   2017-08-29
Name: arrival_date, Length: 119210, dtype: datetime64[ns]

## Changing datatypes

In [26]:

df['meal'] = df['meal'].astype(str)
df['country'] = df['country'].astype(str)
df['market_segment'] = df['market_segment'].astype(str)
df['customer_type'] = df['customer_type'].astype(str)


# Key Business Insights

In [27]:
#Average revenue per booking (non-cancelled only)
confirmed= df[df['is_canceled']==0]
print(f"Avg revenue per confirmed booking: £{confirmed['total_revenue'].mean():.2f}")

Avg revenue per confirmed booking: £346.44


In [28]:
#Top 10 countries by bookings
print(df['country'].value_counts().head(10))

country
PRT    48483
GBR    12120
FRA    10401
ESP     8560
DEU     7285
ITA     3761
IRL     3374
BEL     2342
BRA     2222
NLD     2103
Name: count, dtype: int64


In [29]:
#Overall cancellation rate
cancel_rate= df['is_canceled'].mean() *100
print(f"Overall cancellation rate: {cancel_rate:.1f}%")

Overall cancellation rate: 37.1%


In [30]:
# Cancellation rate by hotel
print(df.groupby('hotel')['is_canceled'].mean() * 100)

hotel
City Hotel      41.785935
Resort Hotel    27.767373
Name: is_canceled, dtype: float64


In [31]:
# Cancellation rate by customer type

print(df.groupby('customer_type')['is_canceled'].mean() * 100)

customer_type
Contract           30.992141
Group              10.104530
Transient          40.786356
Transient-Party    25.450415
Name: is_canceled, dtype: float64


# Exporting clean dataset

In [33]:
# Export cleaned data for Power BI
df.to_csv('hotel_bookings_clean.csv', index=False)
print("File saved successfully!")

File saved successfully!
